# character-forge-v2 on Colab

Standalone smoke test: install ComfyUI + the GGUF custom node + this package on a fresh Colab GPU runtime, download the ~34GB model set, and generate one real image of the seeded `gtee_dev` character.

This tests the mainstream path (Linux + CUDA) -- this project was originally built and verified on Apple Silicon (MPS), which is the unusual case; CUDA is what ComfyUI itself is most commonly run on.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine for the GGUF-quantized weights used here).

Repo: https://github.com/digishgabhawala/character-forge-v2

## 1. Clone the repo

In [ ]:
!git clone https://github.com/digishgabhawala/character-forge-v2.git
%cd character-forge-v2

## 2. Optional: persist model downloads on Google Drive

The model download is a one-time ~34GB pull, but a fresh Colab VM is disposable -- if you'll be running this notebook more than once, mounting Drive avoids re-downloading every session. Skip this cell to just use the disposable VM disk (simpler, but a fresh 34GB download each time).

In [ ]:
USE_DRIVE = False  # set True to persist model downloads across sessions

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELS_DIR = "/content/drive/MyDrive/character-forge-v2-models"
else:
    MODELS_DIR = "/content/ComfyUI/models"

print("Models will be stored at:", MODELS_DIR)

## 3. Install ComfyUI + ComfyUI-GGUF + forge2, download model weights

Takes 15-30+ min depending on Colab's network speed and whether the models are already on Drive from a previous run. Uses `scripts/colab_bootstrap.sh` -- the same script `linkedin-content-agent`'s combined pipeline notebook calls, so there's one source of truth for this setup, not two copies to keep in sync.

In [ ]:
!bash scripts/colab_bootstrap.sh /content/ComfyUI "{MODELS_DIR}"

## 4. Start ComfyUI and wait for it to come up

A notebook cell can't "leave a terminal open" the way local usage does (see the README) -- start it as a background process, then poll `/system_stats` before trying to generate anything.

In [ ]:
import subprocess
import sys

with open("/content/comfyui.log", "w") as comfy_log:
    comfy_proc = subprocess.Popen(
        [sys.executable, "/content/ComfyUI/main.py"],
        stdout=comfy_log, stderr=subprocess.STDOUT,
    )
print("ComfyUI starting (pid", comfy_proc.pid, ") -- log at /content/comfyui.log")

In [ ]:
!python scripts/wait_for_comfyui.py http://127.0.0.1:8188 240

If the above times out: check `/content/comfyui.log` for the real error (most likely cause on a T4 is a VRAM/OOM issue during model load -- see the README's Known Issues once this session is done, and consider `!cat /content/comfyui.log` here).

## 5. Generate a real image of the seeded `gtee_dev` character

`gtee_dev`'s hero.png is already committed to this repo (`workspace/gtee_dev/hero.png`) -- no need to run `concept`/`pick-hero` first, this goes straight to `generate`.

In [ ]:
!forge2 generate gtee_dev "waving hello with a big smile, plain background" --seed 42

In [ ]:
from IPython.display import Image, display

display(Image("workspace/gtee_dev/generated/gen_00042.png"))

## What this proves (and doesn't)

**Proves:** the repo is genuinely clonable and runnable by someone who isn't the original author, on a machine that isn't the original author's Mac, on the CUDA path this was never actually run on before.

**Doesn't cover:** the `lightning` speed profile, `concept`/`pick-hero` (hero creation) on this platform -- only `generate` against the already-locked `gtee_dev` hero was exercised here. See the repo's `TESTING.md` for the full picture of what's automated-tested vs. live-verified vs. not yet covered.